<a href="https://colab.research.google.com/github/pachterlab/cellmender/blob/main/benchmarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# try:
#     from cellsweep import denoise_count_matrix
# except ImportError:
#     print("cellsweep not found, installing...")
#     !pip install -U -q cellsweep[analysis]

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import itertools
import yaml
import requests
import matplotlib.pyplot as plt
import anndata as ad
from collections import OrderedDict
import seaborn as sns
import scanpy as sc
from cellsweep import denoise_count_matrix
import cellsweep.utils as cs_utils

cellsweep_dir = os.path.dirname(os.path.abspath(""))
rver_docker_workspace = "/home/ruser/work/cellsweep"

# import importlib
# import cellsweep.celltype_ambient
# importlib.reload(cellsweep.celltype_ambient)
# from cellsweep.celltype_ambient import denoise_count_matrix

# import importlib
# import cellsweep.utils.visualization_utils as visualization_utils
# importlib.reload(visualization_utils)
# importlib.reload(cs_utils)

# Compare CellBender vs. cellsweep

Some datasets of use:
- pbmc8k: 8k PBMCs from a healthy donor (CellBender Fig2): https://www.10xgenomics.com/datasets/8-k-pbm-cs-from-a-healthy-donor-2-standard-2-1-0
  - see run configuration on page 13 (bottom left) of the [Cellbender manuscript](https://doi.org/10.1038/s41592-023-01943-7)
- hgmm12k: Human-mouse mixture (CellBender Fig5): https://support.10xgenomics.com/single-cell-gene-expression/datasets/2.1.0/hgmm_12k

In [ ]:
dataset_name = "8cubed"
alternative_tools = []  # ["cellbender", "soupx", "decontx", "scar"]
rerun_tools = False  # rerun tools if files don't exist OR if overwrite=True (else download from Box) - if True, requires docker
verbose = 2  # 2 debug, 1 info, 0 warning, -1 error, -2 critical
overwrite = False  # overwrite existing files
scar_env = "/home/jrich/miniconda3/envs/scar"
use_cuda = False  # for CellBender and scAR
threads = 8  # for cellsweep and CellBender (if use_cuda=False)

In [ ]:
# load yaml
yaml_file = os.path.join(cellsweep_dir, "notebooks", "config", f"{dataset_name}.yaml")
cfg = cs_utils.load_dataset_yaml(yaml_file)

# create directories for data, output
data_dir = os.path.join(cellsweep_dir, "notebooks", "data", dataset_name)
os.makedirs(data_dir, exist_ok=True)

out_dir = os.path.join(cellsweep_dir, "notebooks", "output", dataset_name)
os.makedirs(out_dir, exist_ok=True)

# set file paths for tools
out_dir_cellsweep = os.path.join(data_dir, "cellsweep")
out_dir_cellbender = os.path.join(data_dir, "cellbender")
out_dir_soupx = os.path.join(data_dir, "soupx")
out_dir_decontx = os.path.join(data_dir, "decontx")
out_dir_scar = os.path.join(data_dir, "scar")

# lay out parameters
model_pkl = cfg["model_pkl"]
celltypist_convert = cfg["celltypist_convert"]
celltypist_map_file = cfg["celltypist_map_file"]
cellsweep_max_iter = cfg["cellsweep_max_iter"]
cellsweep_beta = cfg["cellsweep_beta"]
cellsweep_init_alpha = cfg["cellsweep_init_alpha"]
cellbender_epochs = cfg["cellbender_epochs"]
cellbender_fpr = cfg["cellbender_fpr"]
cellbender_zdim = cfg["cellbender_zdim"]
cellbender_expected_cells = cfg["cellbender_expected_cells"]
cellbender_total_droplets = cfg["cellbender_total_droplets"]
scar_epochs = cfg["scar_epochs"]
scar_prob = cfg["scar_prob"]
expected_cells = cfg["expected_cells"]
min_genes = cfg["min_genes"]
min_cells = cfg["min_cells"]
umi_top_percentile_to_remove = cfg["umi_top_percentile_to_remove"]
fraction_doublet = cfg["fraction_doublet"]
unique_genes_top_percentile_to_remove = cfg["unique_genes_top_percentile_to_remove"]
mt_gene_percentile_to_remove = cfg["mt_gene_percentile_to_remove"]
max_mt_percentage = cfg["max_mt_percentage"]
n_top_genes = cfg["n_top_genes"]
n_pcs = cfg["n_pcs"]
n_neighbors = cfg["n_neighbors"]
leiden_resolution = cfg["leiden_resolution"]
marker_genes = cfg["marker_genes"]

adata_raw_path_dict = {
    "Adrenal": "Adrenal_annotated.h5ad",
    "Gastrocnemius": "Gastrocnemius_annotated.h5ad",
    "GonadsMale": "GonadsMale_annotated.h5ad",
    "HypothalamusPituitary": "HypothalamusPituitary_annotated.h5ad",
    "Liver": "Liver_annotated.h5ad",
    "CortexHippocampus": "CortexHippocampus_annotated.h5ad",
    "GonadsFemale": "GonadsFemale_annotated.h5ad",
    "Heart": "Heart_annotated.h5ad",
    "Kidney": "Kidney_annotated.h5ad"
}
adata_raw_path_dict = {tissue: os.path.join(data_dir, filename) for tissue, filename in adata_raw_path_dict.items()}

## Raw

In [ ]:
adata_raw_list = cs_utils.load_adata(cfg["adata_dir"], multiple_anndatas=True, verbose=verbose)
adata_raw_dict = {}
for adata_raw in adata_raw_list:
    adata_raw.var_names_make_unique()
    tissue = adata_raw.obs["Tissue"].unique()[0]
    adata_raw_dict[tissue] = adata_raw
eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")

## Knee plot - use this output to estimate umi_cutoff

In [ ]:
umi_cutoffs = {}
for tissue, adata_raw in adata_raw_dict.items():
    umi_cutoff = cs_utils.knee_plot(adata_raw, expected_cells=expected_cells, title=f"Knee Plot - {tissue}", out_path=os.path.join(out_dir, f"knee_plot_{tissue}.png"))
    umi_cutoffs[tissue] = umi_cutoff

In [ ]:
#!!! optionally update umi_cutoffs from knee plot
adata_raw_filtered_dict = {}
for tissue, adata_raw in adata_raw_dict.items():
    adata_raw = cs_utils.infer_empty_droplets(adata_raw, method="threshold", umi_cutoff=umi_cutoffs[tissue], verbose=verbose)  # adds adata.obs["is_empty"]
    adata_raw.var['empty_counts'] = np.array(adata_raw.X[adata_raw.obs['is_empty'].values, :].sum(axis=0)).flatten()
    adata_raw_dict[tissue] = adata_raw
    adata_raw_filtered = adata_raw[~adata_raw.obs["is_empty"]].copy()
    adata_raw_filtered_dict[tissue] = adata_raw_filtered

## cellsweep

In [ ]:
adata_cellsweep_dict = {}
for tissue, adata_raw in adata_raw_dict.items():
    adata_path_cellsweep = os.path.join(out_dir_cellsweep, f"{tissue}_cellsweep.h5ad")
    if not os.path.exists(adata_path_cellsweep) or overwrite:
        adata_cellsweep = denoise_count_matrix(adata_raw, adata_out=adata_path_cellsweep, beta=cellsweep_beta, freeze_ambient_profile=True, init_alpha=cellsweep_init_alpha, max_iter=cellsweep_max_iter, empty_droplet_method="threshold", expected_cells=expected_cells, threads=threads, verbose=verbose, log_file=os.path.join(data_dir, "cellsweep.log"))
    else:
        adata_cellsweep = ad.read_h5ad(adata_path_cellsweep)
    adata_cellsweep = adata_cellsweep[~adata_cellsweep.obs["is_empty"]].copy()
    adata_cellsweep.var_names_make_unique()
    adata_cellsweep_dict[tissue] = adata_cellsweep

# Analysis

In [ ]:
for tissue in adata_raw_dict.keys():
    print(f"Tissue: {tissue}")
    print(f"Adata raw (filtered cells): {adata_raw_filtered_dict[tissue]}")
    print(f"Adata cellsweep: {adata_cellsweep_dict[tissue]}\n\n")

In [ ]:
def concatenate_adatas(adata_list):
    adatas = []
    for adata in adata_list:
        adatas.append(adata)
    adata_combined = ad.concat(adatas, join="outer", label="Tissue", keys=list(adata_raw_filtered_dict.keys()))
    return adata_combined

adata_raw_filtered = concatenate_adatas(adata_raw_filtered_dict.values())
adata_cellsweep = concatenate_adatas(adata_cellsweep_dict.values())

adata_dict = {
    "Raw": adata_raw_filtered,
    "cellsweep": adata_cellsweep
}

## Save Anndatas

In [ ]:
for tool, adata in adata_dict.items():
    adata_path = os.path.join(data_dir, tool, f"combined_{tool}.h5ad")
    os.makedirs(os.path.dirname(adata_path), exist_ok=True)
    adata.write_h5ad(adata_path)

## 8 Cubed Analysis

In [ ]:
# data_dir = os.path.join(cellsweep_dir, "notebooks", "data", "8cubed")
# adata_dict = {}
# for tool in ["raw", "cellsweep"]:
#     adata_path = os.path.join(data_dir, tool, f"combined_{tool}.h5ad")
#     adata_dict[tool] = ad.read_h5ad(adata_path)
# eight_cubed_markers_path = os.path.join(data_dir, "8_cube_marker_genes.csv")
# out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "8cubed")

cs_utils.make_8cubed_plots(adata_dict, eight_cubed_markers_path, out_dir=out_dir)